# Shark Sentinel - Entraînement aerial sur GPU (Colab)

1. Menu **Exécution > Modifier le type d'exécution > GPU (T4)** avant de lancer.
2. Exécute les cellules dans l'ordre.
3. La cellule 2 te demandera d'uploader `shark_sentinel_aerial.zip`.

In [ ]:
!pip install -q ultralytics
import torch
print("GPU dispo :", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "aucun")

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()  # sélectionne shark_sentinel_aerial.zip
zip_name = list(uploaded.keys())[0]

os.makedirs("/content/shark_sentinel_aerial", exist_ok=True)
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall("/content/shark_sentinel_aerial")
print("Extrait dans /content/shark_sentinel_aerial")

In [ ]:
# Corrige le chemin absolu dans data.yaml (celui d'origine pointe vers ta machine Windows)
import yaml

data_yaml_path = "/content/shark_sentinel_aerial/data.yaml"
with open(data_yaml_path) as f:
    data = yaml.safe_load(f)
data["path"] = "/content/shark_sentinel_aerial"
with open(data_yaml_path, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)
print(data)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
model.train(
    data=data_yaml_path,
    epochs=80,
    imgsz=640,
    batch=16,
    device=0,
    project="/content/runs/shark_sentinel_aerial",
    name="train",
    patience=20,
)

In [ ]:
# Télécharge le modèle entraîné (best.pt) sur ta machine
from google.colab import files
files.download("/content/runs/shark_sentinel_aerial/train/weights/best.pt")